# 03 — Preprocesamiento y reducción de dimensionalidad

Fase 3: codificación, escalado y reducción de dimensionalidad.

## Carga y revisión inicial

Partimos del archivo original para que los cambios queden separados del EDA. Primero identificamos la columna de consecutivo y revisamos duplicados usando las variables clínicas, sin contar ese identificador. Según el análisis de calidad no había duplicados, pero se vuelve a comprobar antes de transformar.

Los valores atípicos detectados en el EDA no se eliminan: pueden corresponder a resultados clínicos reales y no se identificaron como errores.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

rutas_candidatas = [
    Path("../data/raw/hcvdat0.csv"),
    Path("data/raw/hcvdat0.csv"),
]
ruta_hcv = next((ruta for ruta in rutas_candidatas if ruta.exists()), None)
if ruta_hcv is None:
    raise FileNotFoundError("No se encontró data/raw/hcvdat0.csv.")

df_original = pd.read_csv(ruta_hcv)
columna_id = df_original.columns[0]
columnas_datos = df_original.columns.drop(columna_id)

# El ID no describe al paciente; se excluye al buscar duplicados clínicos.
duplicados_clinicos = df_original.duplicated(
    subset=columnas_datos, keep="first"
)
print(f"Registros originales: {len(df_original)}")
print(f"Duplicados clínicos encontrados: {duplicados_clinicos.sum()}")

df_limpio = df_original.loc[~duplicados_clinicos].drop(columns=columna_id).copy()
print(f"Registros después de revisar duplicados: {len(df_limpio)}")
print(f"Columnas conservadas, sin identificador: {df_limpio.shape[1]}")
display(df_limpio.head())

### Tratamiento de faltantes

El EDA encontró 31 valores faltantes en 26 filas. Como la mayoría está en mediciones clínicas y `ALP` tiene faltantes concentrados en grupos de pacientes, no borramos esas filas. Para cada variable numérica usamos la mediana de su grupo `Category`; si algún grupo no tiene valores disponibles, usamos como respaldo la mediana general de esa variable.

También guardamos un indicador de faltante para `ALP`, porque en el EDA se observó que su ausencia se concentra en categorías clínicas. Esto permite conservar esa información después de imputar.

In [ ]:
faltantes_antes = df_limpio.isna().sum()
if "ALP" in df_limpio.columns:
    df_limpio["ALP_faltante"] = df_limpio["ALP"].isna().astype("int8")

columnas_numericas = df_limpio.select_dtypes(include="number").columns.tolist()
columnas_imputadas = [
    columna for columna in columnas_numericas
    if df_limpio[columna].isna().any()
]

for columna in columnas_imputadas:
    mediana_general = df_limpio[columna].median()
    medianas_por_categoria = df_limpio.groupby(
        "Category", observed=False
    )[columna].transform("median")
    df_limpio[columna] = (
        df_limpio[columna]
        .fillna(medianas_por_categoria)
        .fillna(mediana_general)
    )

resumen_faltantes = pd.DataFrame({
    "Faltantes_antes": faltantes_antes,
    "Faltantes_despues": df_limpio.isna().sum(),
})
resumen_faltantes = resumen_faltantes.query(
    "Faltantes_antes > 0 or Faltantes_despues > 0"
)
display(resumen_faltantes)
print(f"Filas conservadas: {len(df_limpio)}")
print(f"Columnas imputadas: {columnas_imputadas}")
assert not df_limpio.isna().any().any(), "Quedaron valores faltantes sin tratar."

### Transformación y codificación

En el EDA se encontró sesgo positivo fuerte en `ALP`, `ALT`, `AST`, `BIL`, `CREA` y `GGT`. La prueba con `log1p` redujo ese sesgo, por eso aplicamos la transformación a esas seis variables después de imputar. No transformamos los otros marcadores, porque en ellos el logaritmo no mejoró la distribución.

`Category` y `Sex` son variables nominales, así que usamos One-Hot Encoding y no Label Encoding. Conservamos la categoría sospechosa como una clase independiente: con la información revisada no hay sustento para mezclarla con otra. La etiqueta de diagnóstico queda representada en columnas propias; esas columnas identifican la clase y no deben usarse como mediciones de entrada en un modelo.

In [ ]:
df_modelo = df_limpio.copy()
categorias_diagnostico = df_modelo["Category"].copy()

variables_log = [
    "ALP", "ALT", "AST", "BIL", "CREA", "GGT",
]
variables_log = [
    columna for columna in variables_log if columna in df_modelo.columns
]

# log1p es válido porque el EDA no encontró mediciones clínicas no positivas.
for columna in variables_log:
    if (df_modelo[columna] < 0).any():
        raise ValueError(f"{columna} contiene valores negativos; no se aplica log1p.")
    df_modelo[f"log1p_{columna}"] = np.log1p(df_modelo[columna])
    df_modelo = df_modelo.drop(columns=columna)

columnas_categoricas = df_modelo.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
df_procesado = pd.get_dummies(
    df_modelo,
    columns=columnas_categoricas,
    prefix=columnas_categoricas,
    dtype="int8",
)

print(f"Variables transformadas con log1p: {variables_log}")
print(f"Variables categóricas codificadas: {columnas_categoricas}")
print(f"Dimensiones del dataset procesado: {df_procesado.shape}")
print(f"¿Todas las columnas son numéricas?: {df_procesado.dtypes.apply(pd.api.types.is_numeric_dtype).all()}")
display(df_procesado.head())

In [ ]:
# Verificaciones simples antes de guardar el resultado del preprocesamiento.
assert not df_procesado.isna().any().any(), "El dataset procesado tiene faltantes."
assert df_procesado.dtypes.apply(pd.api.types.is_numeric_dtype).all(), (
    "El dataset procesado tiene columnas no numéricas."
)

ruta_proyecto = ruta_hcv.resolve().parents[2]
ruta_salida = ruta_proyecto / "data" / "processed" / "hcv_processed.csv"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
df_procesado.to_csv(ruta_salida, index=False)

print(f"Archivo procesado guardado en: {ruta_salida}")
print(f"Valores faltantes finales: {int(df_procesado.isna().sum().sum())}")
print(f"Filas finales: {len(df_procesado)}")
print(
    "La serie `categorias_diagnostico` conserva la etiqueta original para "
    "interpretar o graficar los grupos en las siguientes etapas."
)